# 01 - Verify ingestion of `ai_narratives_original`

This notebook is the integrity check for the analysis schema.
Run it whenever you re-populate the schema to confirm row counts and basic shape.

**Ingestion entry points** (these populate the schema; do *not* run from inside this notebook):
- `bash scripts/migrate/run_all.sh` - end-to-end
- `scripts/migrate/ingest_real_from_xlsx.py` - real-patient data from `data/Data_real_sample.xlsx`
- `scripts/migrate/ingest_llm_from_db.py` - LLM synthetic for all models
- `scripts/migrate/ingest_expert_feedback.py` - expert Likert ratings

## 1. Setup

In [1]:
import sys
from pathlib import Path
REPO = Path.cwd().parent
sys.path.insert(0, str(REPO / 'src'))
from sqlalchemy import text
from pain_narratives.core.database import DatabaseManager
from pain_narratives.analysis import revision_data_layer as dl

SCHEMA = 'ai_narratives_original'
db = DatabaseManager()
print(f'Connected. Schema under inspection: {SCHEMA}')


Connected. Schema under inspection: ai_narratives_original


## 2. Row counts per table

In [2]:
tables = ('narratives',
          'real_patient_demographics','real_patient_pcs',
          'real_patient_bpi','real_patient_tsk',
          'llm_dimension_evaluation','llm_pcs_results',
          'llm_bpi_results','llm_tsk_results',
          'expert_dimension_evaluation','expert_questionnaire_feedback',
          'sus_usability_results')
expected = {'narratives': 152,
            'real_patient_demographics': 152,
            'real_patient_pcs': 152,
            'real_patient_bpi': 152,
            'real_patient_tsk': 152,
            'llm_dimension_evaluation': 152*3*3,
            'llm_pcs_results': 152*3*3,
            'llm_bpi_results': 152*3*3,
            'llm_tsk_results': 152*3*3}
print(f'{"table":<34} {"rows":>8}  {"expect":>8}  status')
print('-' * 70)
with db.engine.connect() as conn:
    for t in tables:
        n = conn.execute(text(f'SELECT COUNT(*) FROM {SCHEMA}.{t}')).scalar()
        exp = expected.get(t, '-')
        status = 'OK' if exp == '-' or n >= 0.99 * exp else 'CHECK'
        print(f'  {t:<32} {n:>8}  {str(exp):>8}  {status}')


table                                  rows    expect  status
----------------------------------------------------------------------


  narratives                            152       152  OK
  real_patient_demographics             152       152  OK
  real_patient_pcs                      152       152  OK
  real_patient_bpi                      152       152  OK
  real_patient_tsk                      152       152  OK


  llm_dimension_evaluation             1368      1368  OK
  llm_pcs_results                      1368      1368  OK
  llm_bpi_results                      1367      1368  OK
  llm_tsk_results                      1367      1368  OK
  expert_dimension_evaluation            26         -  OK


  expert_questionnaire_feedback          70         -  OK
  sus_usability_results                   0         -  OK


## 3. Models available in the LLM tables

In [3]:
print('Available models:', dl.available_models())
with db.engine.connect() as conn:
    rows = conn.execute(text(f'''
        SELECT model, run_number, COUNT(*) AS n
        FROM {SCHEMA}.llm_dimension_evaluation
        GROUP BY model, run_number ORDER BY model, run_number
    ''')).fetchall()
for m, r, n in rows:
    print(f'  {m:<32} run {r}: {n}')


Available models: ['claude-sonnet-4-5-thinking', 'deepseek-r1', 'gpt-5']
  claude-sonnet-4-5-thinking       run 1: 152
  claude-sonnet-4-5-thinking       run 2: 152
  claude-sonnet-4-5-thinking       run 3: 152
  deepseek-r1                      run 1: 152
  deepseek-r1                      run 2: 152
  deepseek-r1                      run 3: 152
  gpt-5                            run 1: 152
  gpt-5                            run 2: 152
  gpt-5                            run 3: 152


## 4. Real-data baseline (must reproduce the published paper)

In [4]:
real = dl.load_real_from_schema()
print(f'PCS  mean: {real["pcs_total"].mean():.2f} +/- {real["pcs_total"].std():.2f}  (paper: 31.11 +/- 11.70)')
print(f'BPI  mean: {real["bpi_total_mean"].mean():.2f} +/- {real["bpi_total_mean"].std():.2f}  (paper:  7.02 +/-  1.88)')
print(f'TSK  mean: {real["tsk_total"].mean():.2f} +/- {real["tsk_total"].std():.2f}  (paper: 28.28 +/-  6.27)')


PCS  mean: 31.44 +/- 11.71  (paper: 31.11 +/- 11.70)
BPI  mean: 6.90 +/- 1.68  (paper:  7.02 +/-  1.88)
TSK  mean: 28.30 +/- 6.26  (paper: 28.28 +/-  6.27)


## 5. Sample narratives

In [5]:
with db.engine.connect() as conn:
    rows = conn.execute(text(f'''
        SELECT n.narrative_id, n.word_count, LEFT(n.narrative_text, 80) AS preview
        FROM {SCHEMA}.narratives n
        ORDER BY n.narrative_id LIMIT 5
    ''')).fetchall()
for nid, wc, preview in rows:
    print(f'  #{nid:>3} ({wc:>4}w) {preview!r}')


  #  1 (  63w) 'Lo recuerdo desde la infancia.\nEs un dolor generalizado, persistente, cuando más'
  #  2 (  92w) 'Mis dolores empezaron en alguna zona en concreto y con cansancio.Pero poco a poc'
  #  3 (  39w) 'Mi vida desde los 48 años ya no es la misma yo me comía el mundo y ahora el mund'
  #  4 ( 118w) 'A ver pues mi dolor empezó fuerte fuerte el año de nacimiento 2019 mi hijo.Empez'
  #  5 ( 485w) 'Actualmente (17/09/25), he tenido un dolor muy intenso desde anoche. Ayer por la'


## 6. Done

In [6]:
print('All checks complete.')
print('To refresh data, run: bash scripts/migrate/run_all.sh')


All checks complete.
To refresh data, run: bash scripts/migrate/run_all.sh
